# VLM Hallucination — Mechanistic Analysis

## Investigating Why Vision-Language Models Hallucinate Objects

**Target Model:** `liuhaotian/llava-v1.5-7b`  
**Dataset:** POPE Adversarial (3000 Yes/No questions, 500 images)  
**Framework:** PyTorch hooks + HuggingFace Transformers  
**Environment:** Kaggle GPU (T4 16GB)

This notebook adapts mechanistic interpretability techniques — originally applied to text-only
LLMs — to multimodal vision-language hallucination:

| # | Experiment | Key Question |
|---|-----------|-------------|
| E1 | Per-Layer **Logit Lens** | In which layers do hallucinations vs correct predictions diverge? |
| E2 | **VCD Noise Probing** | Which layers are most sensitive to visual perturbation? |
| E3 | **Visual Logit Lens** (CVPR 2026) | What does the model "see" in high-attention image regions? |
| E4 | **Activation Patching** | Can we causally probe the vision-to-language pathway? |
| E5 | **DoLa Layer Contrasting** | Does early-layer logit subtraction suppress language-prior bias? |

**Core Metric:**  `logit_diff = logit("Yes") − logit("No")`  
**Baselines:**  VCD Acc 80.0%  |  DoLa Acc 83.5% (on POPE Adversarial)

---
**References:**
- VCD: Leng et al., CVPR 2024 — Visual Contrastive Decoding
- DoLa: Chuang et al., ICLR 2024 — Decoding by Contrasting Layers
- Wang et al., CVPR 2026 — Logit-Lens over Visual Attention (SADT)
- Wang et al., 2022 — Interpretability in the Wild (IOI Circuit)

## Section 0: Environment Setup & Imports

### S0.1 Install Dependencies

Run once at the top of Kaggle session. Will skip if packages already exist.

In [ ]:
import sys, subprocess, importlib

DEPS = {
    "transformers":  None,   # latest
    "accelerate":    None,
    "bitsandbytes":  None,
    "sentencepiece": None,
}

for pkg in DEPS:
    modname = pkg.replace("-", "_")
    try:
        importlib.import_module(modname)
        print(f"  ✓ {pkg} already installed")
    except ImportError:
        print(f"  Installing {pkg} ...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

print("Done.")

In [ ]:
import os, sys, json, math
from pathlib import Path
from collections import defaultdict
from functools import partial
import warnings
warnings.filterwarnings("ignore")

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch import Tensor

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from tqdm.auto import tqdm

# ── Paths ──
WORKSPACE_ROOT = Path.cwd().resolve()
VCD_ROOT       = WORKSPACE_ROOT / "VCD"
EXP_ROOT       = VCD_ROOT / "experiments"
LLAVA_ROOT     = EXP_ROOT / "llava"
DOLA_ROOT      = WORKSPACE_ROOT / "DoLa"
DATA_DIR       = WORKSPACE_ROOT / "data"
RESULTS_DIR    = WORKSPACE_ROOT / "results"

for p in [str(VCD_ROOT), str(EXP_ROOT), str(LLAVA_ROOT), str(DOLA_ROOT)]:
    if p not in sys.path:
        sys.path.insert(0, p)

print(f"Workspace: {WORKSPACE_ROOT}")
print(f"Data dir:  {DATA_DIR}")
print(f"CUDA:      {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU:       {torch.cuda.get_device_name(0)}")
    ram_gb = torch.cuda.get_device_properties(0).total_mem / 1e9
    print(f"VRAM:      {ram_gb:.1f} GB")

In [ ]:
# ── Color palette for model-prediction categories (CVD-safe) ──
CAT_COLORS = {
    "TP": "#2a78d6",   # blue — Correct "Yes"
    "TN": "#4caf50",   # green — Correct "No"
    "FP": "#d32f2f",   # red — HALLUCINATION (says Yes, should say No)
    "FN": "#ff9800",   # amber — Missed detection (says No, should say Yes)
}

# ── Token ID helpers ──
def get_yes_no_ids(tokenizer):
    """Return (yes_id, no_id) from tokenizer."""
    yes_id = tokenizer.encode("Yes", add_special_tokens=False)[-1]
    no_id  = tokenizer.encode("No",  add_special_tokens=False)[-1]
    return yes_id, no_id

def logit_diff(logits, yes_id, no_id):
    """logit("Yes") - logit("No"). Works on any shape ending in [vocab_size]."""
    return logits[..., yes_id] - logits[..., no_id]

# ── Logit Lens projection from hidden state ──
def project_logit_diff(hidden, lm_head, yes_id, no_id, ln=None):
    """Apply Logit Lens: project hidden through (optional) LayerNorm + W_U
    and compute logit diff.

    param hidden: [*dims, d_model]
    param lm_head: nn.Linear or weight tensor [vocab, d_model]
    param ln:      LayerNorm module or None
    returns:       [*dims] logit_diff
    """
    if ln is not None:
        hidden = ln(hidden)
    W = lm_head.weight if hasattr(lm_head, 'weight') else lm_head
    logits = hidden @ W.T
    return logits[..., yes_id] - logits[..., no_id]

# ── Activation cache via register_forward_hook ──
class ActCache:
    """Capture hidden states from specific LLaMA decoder layers."""
    def __init__(self):
        self.data = {}    # key -> Tensor
        self.handles = []

    def hook_layer(self, model, layer_id):
        """Save the output hidden_states of one layer."""
        def fn(m, inp, out):
            # LLaMA decoder layer output: out[0] = hidden_states (batch, seq, dim)
            self.data[f"L{layer_id}"] = out[0].detach().cpu()
        ll = model.model.layers[layer_id]
        h = ll.register_forward_hook(fn)
        self.handles.append(h)

    def remove(self):
        for h in self.handles:
            h.remove()
        self.handles.clear()

print("✓ Utilities ready.")

## Section 1: Model Loading & Architecture Reconnaissance

Load the LLaVA-1.5-7B model using the official modules from the VCD repository.
We'll map the full architecture (vision tower, projector, LLaMA backbone) and
verify the forward pass.

In [ ]:
# ── Import LLaVA-specific modules from VCD repo ──
from llava.constants import IMAGE_TOKEN_INDEX, DEFAULT_IMAGE_TOKEN
from llava.conversation import conv_templates
from llava.mm_utils import tokenizer_image_token, get_model_name_from_path
from llava.model.builder import load_pretrained_model

MODEL_PATH = "liuhaotian/llava-v1.5-7b"
MODEL_BASE = "lmsys/vicuna-7b-v1.5"

print(f"Loading {MODEL_PATH} ... (this may take a few minutes)")

tokenizer, model, image_processor, context_len = load_pretrained_model(
    model_path=MODEL_PATH,
    model_base=MODEL_BASE,
    model_name=get_model_name_from_path(MODEL_PATH),
    load_8bit=False,
    load_4bit=False,   # float16 on GPU
)

model.eval()
print(f"✓ Model loaded. Context length: {context_len}")

In [ ]:
# ── Architecture inventory ──
vt      = model.get_vision_tower()        # CLIP ViT-L/14
proj    = model.model.mm_projector        # linear(1024 -> 4096)
lm_head = model.lm_head                   # unembedding matrix
backbone = model.model                    # LlamaModel(LlamaDecoderLayer x 32)

N_LAYERS   = len(backbone.layers)
HIDDEN_DIM = model.config.hidden_size
VOCAB_SIZE = model.config.vocab_size

# Determine image token count at runtime
device = next(model.parameters()).device
dtype  = model.dtype
dummy_img = torch.randn(1, 3, 336, 336, device=device, dtype=dtype)

with torch.inference_mode():
    v_out = vt(dummy_img)           # [1, 577, 1024]
    p_out = proj(v_out)             # [1, 576, 4096] (excludes CLS)
    N_IMG_TOKENS = p_out.shape[1]

print("=" * 55)
print("LLaVA-1.5-7B  ARCHITECTURE")
print("=" * 55)
print(f" Vision Tower:           CLIP ViT-L/14 (336x336)")
print(f"   raw CLIP tokens:        {v_out.shape[1]}")
print(f"   after projector:        {N_IMG_TOKENS} image tokens")
print(f"   projector:              linear({vt.hidden_size} -> {HIDDEN_DIM})")
print(f"")
print(f" Language Backbone:       LLaMA-7B")
print(f"   layers:                 {N_LAYERS}")
print(f"   d_model:                {HIDDEN_DIM}")
print(f"   vocab_size:             {VOCAB_SIZE}")
print(f"   lm_head.weight:         [{VOCAB_SIZE}, {HIDDEN_DIM}]")
print("=" * 55)

# Store core handles for experiments
FINAL_LN = model.model.norm         # final LayerNorm before lm_head
LM_HEAD  = model.model_head
NUM_LAYERS = N_LAYERS
N_IMG = N_IMG_TOKENS